In [9]:
from sklearn.utils import shuffle
from numpy import dtype
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

california = fetch_california_housing(as_frame=False)
x,y = california.data,california.target
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=.2,random_state=42)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.fit_transform(x_test)

device = torch.device ('cuda' if torch.cuda.is_available() else 'cpu')
x_train_t = torch.tensor(x_train,dtype=torch.float32)
y_train_t = torch.tensor(y_train,dtype=torch.float32).unsqueeze(1)
x_test_t = torch.tensor(x_test,dtype = torch.float32)
y_test_t = torch.tensor(y_test,dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(x_train_t,y_train_t),batch_size=64,shuffle=True,drop_last=True)
test_loader = DataLoader(TensorDataset(x_test_t,y_test_t),batch_size=64,shuffle=False)

print(f"Processing on: {device}")
print(f"Train Shape: {x_train_t.shape} | Test Shape: {x_test_t.shape}")


Processing on: cuda
Train Shape: torch.Size([16512, 8]) | Test Shape: torch.Size([4128, 8])


In [14]:
class WideAndDeepMLP(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.hidden1 = nn.Linear(num_features, 64)
        self.hidden2 = nn.Linear(64, 64)
        self.relu = nn.ReLU()
        
        self.dropout = nn.Dropout(p=0.3) 
        
        self.wide_path = nn.Linear(num_features, 16) 
        
        self.output_layer = nn.Linear(64 + 16, 1)

    def forward(self, inputs):
        deep = self.relu(self.hidden1(inputs))
        deep = self.dropout(deep)
        deep = self.relu(self.hidden2(deep))
        deep = self.dropout(deep)
        
        wide = self.wide_path(inputs)
        
        combined = torch.cat([deep, wide], dim=1)
        return self.output_layer(combined)
model = WideAndDeepMLP(num_features=8).to(device)
print(model)

WideAndDeepMLP(
  (hidden1): Linear(in_features=8, out_features=64, bias=True)
  (hidden2): Linear(in_features=64, out_features=64, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (wide_path): Linear(in_features=8, out_features=16, bias=True)
  (output_layer): Linear(in_features=80, out_features=1, bias=True)
)


In [16]:
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-3, weight_decay=1e-2)

EPOCHS = 30
print(" Starting Training on California Housing Dataset...\n")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        running_loss += loss.item()
        
    model.eval()
    running_test_mse = 0.0
    running_test_mae = 0.0
    
    with torch.no_grad():
        for test_X, test_y in test_loader:
            test_X, test_y = test_X.to(device), test_y.to(device)
            preds = model(test_X)
            
            running_test_mse += criterion(preds, test_y).item()
            running_test_mae += torch.mean(torch.abs(preds - test_y)).item()

    epoch_train_mse = running_loss / len(train_loader)
    epoch_test_mse  = running_test_mse / len(test_loader)
    epoch_test_mae  = running_test_mae / len(test_loader)
    
    actual_dollar_error = epoch_test_mae * 100000 
    
    print(f" Epoch [{epoch+1}/{EPOCHS}]")
    print(f"   ↳ Train MSE : {epoch_train_mse:.4f}")
    print(f"   ↳ Test MSE  : {epoch_test_mse:.4f}")
    print(f"   ↳ Avg Error : ${actual_dollar_error:,.2f} away from true price")
    print("-" * 45)

 Starting Training on California Housing Dataset...

 Epoch [1/30]
   ↳ Train MSE : 0.3320
   ↳ Test MSE  : 5.1844
   ↳ Avg Error : $140,043.34 away from true price
---------------------------------------------
 Epoch [2/30]
   ↳ Train MSE : 0.3714
   ↳ Test MSE  : 2.1458
   ↳ Avg Error : $95,996.40 away from true price
---------------------------------------------
 Epoch [3/30]
   ↳ Train MSE : 0.3259
   ↳ Test MSE  : 5.3461
   ↳ Avg Error : $142,997.13 away from true price
---------------------------------------------
 Epoch [4/30]
   ↳ Train MSE : 0.3265
   ↳ Test MSE  : 4.4134
   ↳ Avg Error : $130,105.99 away from true price
---------------------------------------------
 Epoch [5/30]
   ↳ Train MSE : 0.3259
   ↳ Test MSE  : 3.2955
   ↳ Avg Error : $114,807.61 away from true price
---------------------------------------------
 Epoch [6/30]
   ↳ Train MSE : 0.3278
   ↳ Test MSE  : 3.2534
   ↳ Avg Error : $113,569.91 away from true price
---------------------------------------------
